In [85]:
import pyspark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType 

In [70]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

1. What's the sparkSession version?

In [30]:
spark.version

'3.3.2'

In [31]:
# Downloaa data
#!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

In [32]:
df = spark.read.parquet('yellow_tripdata_2024-10.parquet')

In [33]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [34]:
#df = df.repartition(4)

In [35]:
#df.write.parquet('yellow_tripdata/2024/10/')

2. What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

In [36]:
!ls -l ./yellow_tripdata/2024/10/ --block-size=M

total 97M
-rw-r--r-- 1 sebas sebas  0M Mar 15 20:36 _SUCCESS
-rw-r--r-- 1 sebas sebas 25M Mar 15 20:36 part-00000-b1f8cacf-eb5e-434f-9296-20495ebbfa75-c000.snappy.parquet
-rw-r--r-- 1 sebas sebas 25M Mar 15 20:36 part-00001-b1f8cacf-eb5e-434f-9296-20495ebbfa75-c000.snappy.parquet
-rw-r--r-- 1 sebas sebas 25M Mar 15 20:36 part-00002-b1f8cacf-eb5e-434f-9296-20495ebbfa75-c000.snappy.parquet
-rw-r--r-- 1 sebas sebas 25M Mar 15 20:36 part-00003-b1f8cacf-eb5e-434f-9296-20495ebbfa75-c000.snappy.parquet


3. How many taxi trips were there on the 15th of October?

In [37]:
df.select('VendorID','tpep_pickup_datetime') \
    .filter(df.tpep_pickup_datetime >= '2024-10-15').filter(df.tpep_pickup_datetime < '2024-10-16') \
    .sort('tpep_pickup_datetime') \
    .count()

128893

4. What is the length of the longest trip in the dataset in hours?

In [50]:
df = df.withColumn(
    "date_diff_hour",
    (F.col("tpep_dropoff_datetime") - F.col("tpep_pickup_datetime")).cast("long")/60/60.)

In [57]:
df.registerTempTable('yellow_trip')

/home/sebas/spark/spark-3.3.2-bin-hadoop3/python/pyspark/sql/dataframe.py:229: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [64]:
spark.sql("""
    SELECT MAX(date_diff_hour) longest_trip
    FROM yellow_trip
    LIMIT 1;
    """).show()

+-----------------+
|     longest_trip|
+-----------------+
|162.6177777777778|
+-----------------+



5. Spark’s User Interface which shows the application's dashboard runs on which local port?

In [65]:
# 4040

6. Least frequent pickup location zone

In [66]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-15 21:56:41--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.33.163.58, 13.33.163.188, 13.33.163.101, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.33.163.58|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2025-03-15 21:56:41 (33.6 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [83]:
schema = StructType([ 
    StructField('LocationID', 
                IntegerType(), True), 
    StructField('Borough', 
                StringType(), True), 
    StructField('Zone', 
                StringType(), True), 
    StructField('service_zone', 
                StringType(), True)
]) 
  
# Applying custom schema to data frame 
df_zone = spark.read.format( 
    "csv").schema(schema).option( 
    "header", True).load("taxi_zone_lookup.csv") 
df_zone.registerTempTable('taxi_zone')

/home/sebas/spark/spark-3.3.2-bin-hadoop3/python/pyspark/sql/dataframe.py:229: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [88]:
df_zone.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [87]:
spark.sql("""
    SELECT tz.Zone, count(1) cnt
    FROM yellow_trip yt
      LEFT JOIN taxi_zone tz
       ON yt.PULocationID = tz.LocationID
    GROUP BY 1
    ORDER BY cnt ASC
    LIMIT 5;
    """).show()

+--------------------+---+
|                Zone|cnt|
+--------------------+---+
|Governor's Island...|  1|
|       Rikers Island|  2|
|       Arden Heights|  2|
|         Jamaica Bay|  3|
| Green-Wood Cemetery|  3|
+--------------------+---+

